# Feature Engineering on Primary Land Use Tax Lot Output (PLUTO) and Driver Revenue Datasets:

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import geopandas as gpd
from shapely import wkt
import pandas as pd 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_pluto+revenue")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"

PLUTO dataset:

In [ ]:
pluto_df_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_df_path)
pluto_df.head()

Zone dataset:

In [ ]:
zone_gdf_path = base_dir + '/developed/merged_data/zone_gdf.csv'
zone_gdf = pd.read_csv(zone_gdf_path)
zone_gdf['geometry'] = zone_gdf['geometry'].apply(wkt.loads)
zone_gdf.head()

Daily revenue:

In [ ]:
daily_revenue_sdf_path = base_dir + '/developed/merged_data/daily_revenue'
daily_revenue_sdf = spark.read.parquet(daily_revenue_sdf_path)
daily_revenue_df = daily_revenue_sdf.toPandas()
daily_revenue_df.head()

# Find Daily Revenue by Building Class:

In [ ]:
# Merge `pluto_df` and `daily_revenue_df` on `location_id` and `PULocationID`
daily_revenue_by_building_class_df = pd.merge(pluto_df, daily_revenue_df, 
                                              left_on='location_id', 
                                              right_on='PULocationID', 
                                              how='left')

# Group by `building_class`, and calculate the sum of `daily_revenued`
daily_revenue_by_building_class_df = daily_revenue_by_building_class_df.groupby(['building_class'])['daily_revenue'] \
                                                                       .sum() \
                                                                       .reset_index() 

# Sort by daily revenue
daily_revenue_by_building_class_df = daily_revenue_by_building_class_df.sort_values(by='daily_revenue', ascending=False)\
                                                                       .reset_index(drop=True)

daily_revenue_by_building_class_df.head()

In [ ]:
daily_revenue_by_building_class_df.head(24)

# Find Average Daily Revenue by Location ID:

In [ ]:
zone_gdf = gpd.GeoDataFrame(zone_gdf, geometry='geometry')
zone_gdf = zone_gdf.drop_duplicates('location_id')

In [ ]:
# Merge `zone_gdf` and `daily_revenue_df` on `location_id` and `PULocationID`
daily_revenue_by_location_df = pd.merge(zone_gdf, daily_revenue_df, 
                                        left_on='location_id', 
                                        right_on='PULocationID', 
                                        how='left')

# Drop the 'PULocationID' column as it is no longer needed
daily_revenue_by_location_df = daily_revenue_by_location_df.drop('PULocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_revenue`
daily_revenue_by_location_df = daily_revenue_by_location_df.groupby(['location_id'])['daily_revenue'] \
                                                           .sum() \
                                                           .reset_index()

# Sort the DataFrame by `daily_revenue` in descending order
daily_revenue_by_location_df = daily_revenue_by_location_df.sort_values(by='daily_revenue', ascending=False) \
                                                           .reset_index(drop=True)

daily_revenue_by_location_df.head()

In [ ]:
daily_revenue_by_location_df.head(24)

# Save the Merged Dataset:

Daily revenue by different building classes:

In [ ]:
revenue_by_building_class_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue_by_building_class_df.csv'
revenue_by_building_class_df_path = os.path.join(revenue_by_building_class_df_dir, file_name)
daily_revenue_by_building_class_df.to_csv(revenue_by_building_class_df_path, index=False)

Daily revenue by different location ID:

In [ ]:
revenue_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue_by_location_df.csv'
revenue_by_location_df_path = os.path.join(revenue_by_location_df_dir, file_name)
daily_revenue_by_location_df.to_csv(revenue_by_location_df_path, index=False)